# Downloading necessary packages and connecting to MongoDB

In [4]:
# Downloading all necessary packages
import pandas as pd
from pymongo import MongoClient

In [5]:
# Connecting to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["literary_trends"]

# Critic's Pick Cleaning

In [11]:
# Loading critic's picks into a dataframe
critics_data = list(db["critics_picks"].find())
critics_df = pd.DataFrame(critics_data)

print(critics_df.shape)
print(critics_df.columns.tolist())
critics_df.head()

(113549, 20)
['_id', 'abstract', 'web_url', 'snippet', 'lead_paragraph', 'print_section', 'print_page', 'source', 'multimedia', 'headline', 'keywords', 'pub_date', 'document_type', 'news_desk', 'section_name', 'byline', 'type_of_material', 'word_count', 'uri', 'subsection_name']


,_id,abstract,web_url,snippet,lead_paragraph,print_section,print_page,source,multimedia,headline,keywords,pub_date,document_type,news_desk,section_name,byline,type_of_material,word_count,uri,subsection_name
0,nyt://article/0f501db5-3d93-5ea9-afbb-b043b1e5...,Ann Powers reviews performance by rap group Ar...,https://www.nytimes.com/2000/01/01/arts/pop-re...,Ann Powers reviews performance by rap group Ar...,"Q-Unique, one of the rappers in the Bushwick-b...",F,22,The New York Times,[],{'main': 'Rappers Who Won't Let The Audience B...,"[{'name': 'organizations', 'value': 'ARSONISTS...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Ann Powers', 'person': [{'fir...",Review,393,nyt://article/0f501db5-3d93-5ea9-afbb-b043b1e5...,NaN
1,nyt://article/78d0f9c0-3052-59fa-b086-fbd8a3ed...,Jennifer Dunning reviews Eliot Feld Ballet's p...,https://www.nytimes.com/2000/01/01/arts/dance-...,Jennifer Dunning reviews Eliot Feld Ballet's p...,The glorious athletes of Eliot Feld's ''Aurora...,F,10,The New York Times,[],{'main': 'Two Idealized Worlds With Contrastin...,"[{'name': 'organizations', 'value': 'FELD, ELI...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Jennifer Dunning', 'person': ...",Review,330,nyt://article/78d0f9c0-3052-59fa-b086-fbd8a3ed...,NaN
2,nyt://article/af41fef5-ae8c-5120-9711-b1ca69cb...,Jennifer Dunning reviews performance by Alvin ...,https://www.nytimes.com/2000/01/01/arts/dance-...,Jennifer Dunning reviews performance by Alvin ...,One of the happiest aspects of the Alvin Ailey...,F,8,The New York Times,[],{'main': 'Lovers Who Burn the Air Around Them'...,"[{'name': 'organizations', 'value': 'AILEY, AL...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Jennifer Dunning', 'person': ...",Review,350,nyt://article/af41fef5-ae8c-5120-9711-b1ca69cb...,NaN
3,nyt://article/c1e8332c-4e14-5292-86cc-cb1770fb...,Anita Gates reviews play Inappropriate by A Mi...,https://www.nytimes.com/2000/01/01/theater/the...,Anita Gates reviews play Inappropriate by A Mi...,''Inappropriate'' is not ''Rent.'' A theatergo...,F,28,The New York Times,[],"{'main': 'Smells Like Teen Spirit, or Whatever...","[{'name': 'persons', 'value': 'McNeil, Lonnie'...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Theater,"{'original': 'By Anita Gates', 'person': [{'fi...",Review,790,nyt://article/c1e8332c-4e14-5292-86cc-cb1770fb...,NaN
4,nyt://article/c5a5176b-b65d-5a3c-a20e-8391a3f8...,Jack Anderson reviews performance by Forces of...,https://www.nytimes.com/2000/01/01/arts/dance-...,Jack Anderson reviews performance by Forces of...,"Forces of Nature, a company trained in various...",F,18,The New York Times,[],"{'main': 'The Traditions of Africa, Mixed With...","[{'name': 'organizations', 'value': 'Forces of...",2000-01-01T05:00:00+0000,article,The Arts/Cultural Desk,Arts,"{'original': 'By Jack Anderson', 'person': [{'...",Review,275,nyt://article/c5a5176b-b65d-5a3c-a20e-8391a3f8...,NaN


In [19]:
# Keeping only relevant columns
critics_clean = critics_df[[
    'headline',
    'byline',
    'section_name',
    'pub_date',
    'abstract',
    'web_url',
    'keywords'
]].copy()

# Extracting headline text
critics_clean['headline'] = critics_clean['headline'].apply(
    lambda x: x.get('main') if isinstance(x, dict) else x
)

# Extracting byline text
critics_clean['byline'] = critics_clean['byline'].apply(
    lambda x: x.get('original') if isinstance(x, dict) else x
)

# Removing "By " from byline
critics_clean['byline'] = critics_clean['byline'].str.replace('By ', '', regex=False)

# Cleaning up pub_date
critics_clean['pub_date'] = pd.to_datetime(critics_clean['pub_date'], format='mixed').dt.date

# Extracting keywords as comma-separated string
critics_clean['keywords'] = critics_clean['keywords'].apply(
    lambda x: ', '.join([kw.get('value', '') for kw in x]) if isinstance(x, list) else x
)

critics_clean = critics_clean.dropna()
print(f"Critics rows after dropping nulls: {len(critics_clean)}")
critics_clean.head()

Critics rows after dropping nulls: 113548


,headline,byline,section_name,pub_date,abstract,web_url,keywords
0,Rappers Who Won't Let The Audience Be Lazy,Ann Powers,Arts,2000-01-01,Ann Powers reviews performance by rap group Ar...,https://www.nytimes.com/2000/01/01/arts/pop-re...,"ARSONISTS, Music, Reviews"
1,Two Idealized Worlds With Contrasting Moods,Jennifer Dunning,Arts,2000-01-01,Jennifer Dunning reviews Eliot Feld Ballet's p...,https://www.nytimes.com/2000/01/01/arts/dance-...,"FELD, ELIOT, BALLET, Reviews, Dancing"
2,Lovers Who Burn the Air Around Them,Jennifer Dunning,Arts,2000-01-01,Jennifer Dunning reviews performance by Alvin ...,https://www.nytimes.com/2000/01/01/arts/dance-...,"AILEY, ALVIN, AMERICAN DANCE THEATER, Reviews,..."
3,"Smells Like Teen Spirit, or Whatever",Anita Gates,Theater,2000-01-01,Anita Gates reviews play Inappropriate by A Mi...,https://www.nytimes.com/2000/01/01/theater/the...,"McNeil, Lonnie, DeSisto, A Michael, Reviews, T..."
4,"The Traditions of Africa, Mixed With the Modern",Jack Anderson,Arts,2000-01-01,Jack Anderson reviews performance by Forces of...,https://www.nytimes.com/2000/01/01/arts/dance-...,"Forces of Nature, Reviews, Dancing"


# Bestsellers Cleaning

In [17]:
# Loading critic's picks into a dataframe
bestsellers_data = list(db["bestsellers"].find())
bestsellers_df = pd.DataFrame(bestsellers_data)

print(bestsellers_df.shape)
print(bestsellers_df.columns.tolist())
bestsellers_df.head()

(11465, 31)
['_id', 'age_group', 'amazon_product_url', 'article_chapter_link', 'asterisk', 'author', 'book_image', 'book_image_height', 'book_image_width', 'book_review_link', 'book_uri', 'contributor', 'contributor_note', 'created_date', 'dagger', 'description', 'first_chapter_link', 'price', 'primary_isbn10', 'primary_isbn13', 'publisher', 'rank', 'rank_last_week', 'sunday_review_link', 'title', 'updated_date', 'weeks_on_list', 'isbns', 'buy_links', 'list_name', 'list_date']


,_id,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,...,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links,list_name,list_date
0,69dd8e87e0828eb3aa093491,,http://www.amazon.com/Sail-James-Patterson/dp/...,,0,James Patterson and Howard Roughan,https://static01.nyt.com/bestsellers/images/97...,495,273,,...,1,0,,SAIL,2026-03-08T13:17:46.159Z,1,"[{'isbn10': '', 'isbn13': '9780316018708'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2008-07-01
1,69dd8e87e0828eb3aa093492,,http://www.amazon.com/Nothing-Lose-Jack-Reache...,,0,Lee Child,https://static01.nyt.com/bestsellers/images/97...,495,301,,...,2,1,,NOTHING TO LOSE,2026-03-08T13:59:05.751Z,2,"[{'isbn10': '', 'isbn13': '9780385340564'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2008-07-01
2,69dd8e87e0828eb3aa093493,,http://www.amazon.com/The-Host-Novel-Stephenie...,,1,Stephenie Meyer,https://static01.nyt.com/bestsellers/images/97...,495,326,,...,3,2,,THE HOST,2026-03-08T03:57:52.317Z,6,"[{'isbn10': '', 'isbn13': '9780316068048'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2008-07-01
3,69dd8e87e0828eb3aa093494,,http://www.amazon.com/Plague-Ship-The-Oregon-F...,,0,Clive Cussler with Jack Du Brul,https://static01.nyt.com/bestsellers/images/97...,495,275,,...,4,3,,PLAGUE SHIP,2026-03-08T13:59:26.287Z,2,"[{'isbn10': '', 'isbn13': '9780399154973'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2008-07-01
4,69dd8e87e0828eb3aa093495,,http://www.amazon.com/Love-Youre-With-Emily-Gi...,,0,Emily Giffin,https://static01.nyt.com/bestsellers/images/97...,276,185,,...,5,4,,LOVE THE ONE YOU'RE WITH,2026-03-08T15:35:43.757Z,5,"[{'isbn10': '', 'isbn13': '9780312348670'}]","[{'name': 'Amazon', 'url': 'http://www.amazon....",hardcover-fiction,2008-07-01


In [20]:
# Keeping only relevant columns
bestsellers_clean = bestsellers_df[[
    'title',
    'author',
    'publisher',
    'description',
    'rank',
    'weeks_on_list',
    'primary_isbn13',
    'list_name',
    'list_date'
]].copy()

# Clean up list_date to just the date
bestsellers_clean['list_date'] = pd.to_datetime(bestsellers_clean['list_date'], format='mixed').dt.date

bestsellers_clean = bestsellers_clean.dropna()
print(f"Bestsellers rows after dropping nulls: {len(bestsellers_clean)}")
bestsellers_clean.head()

Bestsellers rows after dropping nulls: 11465


,title,author,publisher,description,rank,weeks_on_list,primary_isbn13,list_name,list_date
0,SAIL,James Patterson and Howard Roughan,"Little, Brown",A sailing vacation turns into a disaster when ...,1,1,9780316018708,hardcover-fiction,2008-07-01
1,NOTHING TO LOSE,Lee Child,Delacorte,Jack Reacher exposes the secrets of a Colorado...,2,2,9780385340564,hardcover-fiction,2008-07-01
2,THE HOST,Stephenie Meyer,"Little, Brown",In this first adult novel by the author of the...,3,6,9780316068048,hardcover-fiction,2008-07-01
3,PLAGUE SHIP,Clive Cussler with Jack Du Brul,Putnam,Juan Cabrillo and the crew of the Oregon must ...,4,2,9780399154973,hardcover-fiction,2008-07-01
4,LOVE THE ONE YOU'RE WITH,Emily Giffin,St. Martin’s,A woman's happy marriage is shaken when she en...,5,5,9780312348670,hardcover-fiction,2008-07-01


# Exporting both df as .csv files

In [21]:
critics_clean.to_csv('critics_clean.csv', index=False)
bestsellers_clean.to_csv('bestsellers_clean.csv', index=False)